
## **Sistema de Recomendação de Filmes com SVD**

###  **Objetivo**

Desenvolver um sistema de recomendação baseado em decomposição SVD para prever notas de usuários para filmes não avaliados, utilizando uma matriz de avaliações parcialmente preenchida.

---

### **Contexto**

Você foi contratado por uma plataforma de streaming para implementar um sistema de recomendação. A empresa possui registros das avaliações de **10 usuários** sobre **8 filmes**. No entanto, como é comum, cada usuário avaliou apenas uma fração dos filmes disponíveis.

---

### **Matriz de Avaliações**

A seguir, temos a matriz ( $R \in \mathbb{ℝ}^{10 \times 8}$ ), onde:

* As **linhas** representam usuários ( $U_1$ ) a ( $U_{10}$ )
* As **colunas** representam filmes ( $F_1$ ) a ( $F_8$ )
* Os valores são avaliações de $1$ a $5$, com **$0$** indicando ausência de avaliação

$
R =
\begin{bmatrix}
5 & 3 & 0 & 1 & 0 & 0 & 0 & 2 \\
4 & 0 & 0 & 1 & 2 & 0 & 0 & 0 \\
1 & 1 & 0 & 5 & 0 & 0 & 0 & 0 \\
0 & 0 & 5 & 4 & 0 & 0 & 0 & 0 \\
3 & 0 & 0 & 0 & 4 & 0 & 0 & 5 \\
0 & 2 & 4 & 0 & 0 & 0 & 1 & 0 \\
0 & 0 & 0 & 0 & 0 & 3 & 5 & 2 \\
2 & 0 & 3 & 0 & 0 & 0 & 0 & 4 \\
0 & 0 & 0 & 0 & 0 & 5 & 4 & 0 \\
1 & 2 & 0 & 0 & 0 & 0 & 3 & 0 \\
\end{bmatrix}
$

---

###  **Tarefas**

Você deve:

1. **Pré-processar a matriz:**

   * Substituir os valores ausentes ($0$s) por alguma heurística, como:

     * a média da linha (média por usuário), ou
     * a média da coluna (média por filme)

2. **Aplicar SVD:**

   * Decompor a matriz preenchida em ( $U \Sigma V^T$ )
   * Utilizar **redução de dimensionalidade** com os ( $k = 3$ ) maiores valores singulares

3. **Reconstruir a matriz:**

   * Obter a matriz ( $\hat{R}$ ) com previsões para todas as avaliações
   * Arredondar os valores entre $1$ e $5$, se necessário

4. **Gerar recomendações:**

   * Para cada usuário, identificar os filmes com **maior avaliação prevista** que ele ainda **não avaliou**
   * Sugerir os **top $2$ filmes** para cada usuário

---

###  **Formato Esperado da Saída**

Para cada usuário:

* Avaliações previstas:
  $
  \hat{R}_{i, j} \quad \text{(matriz completa)}
  $
* Recomendações:

  Exemplo:

  ```
  Usuário 1: Recomendamos os filmes F3 (nota prevista 4.8) e F5 (4.2)
  Usuário 2: Recomendamos os filmes F2 (4.6) e F3 (4.1)
  ...
  ```

---

###  Dicas Técnicas

* Para fazer SVD em Python:

  ```python
  from numpy.linalg import svd
  U, S, VT = svd(R_filled, full_matrices=False)
  ```

* Para reconstruir com ( $k$ ) componentes:

  ```python
  k = 3
  S_k = np.diag(S[:k])
  U_k = U[:, :k]
  VT_k = VT[:k, :]
  R_hat = U_k @ S_k @ VT_k
  ```

* Para limitar os valores entre 1 e 5:

  ```python
  R_hat_clipped = np.clip(R_hat, 1, 5)
  ```





In [3]:
import numpy as np
R = np.array([[5, 3, 0, 1, 0, 0, 0, 2],
             [4, 0, 0, 1, 2, 0, 0, 0],
             [1, 1, 0, 5, 0, 0, 0, 0],
             [0, 0, 5, 4, 0, 0, 0, 0],
             [3, 0, 0, 0, 4, 0, 0, 5],
             [0, 2, 4, 0, 0, 0, 1, 0],
             [0, 0, 0, 0, 0, 3, 5, 2],
             [2, 0, 3, 0, 0, 0, 0, 4],
             [0, 0, 0, 0, 0, 5, 4, 0],
             [1, 2, 0, 0, 0, 0, 3, 0]])

# Imputa com a média dos usuários
R_filled = np.array([np.where(r == 0, r[r != 0].mean(), r) for r in R])
R_filled

array([[5.        , 3.        , 2.75      , 1.        , 2.75      ,
        2.75      , 2.75      , 2.        ],
       [4.        , 2.33333333, 2.33333333, 1.        , 2.        ,
        2.33333333, 2.33333333, 2.33333333],
       [1.        , 1.        , 2.33333333, 5.        , 2.33333333,
        2.33333333, 2.33333333, 2.33333333],
       [4.5       , 4.5       , 5.        , 4.        , 4.5       ,
        4.5       , 4.5       , 4.5       ],
       [3.        , 4.        , 4.        , 4.        , 4.        ,
        4.        , 4.        , 5.        ],
       [2.33333333, 2.        , 4.        , 2.33333333, 2.33333333,
        2.33333333, 1.        , 2.33333333],
       [3.33333333, 3.33333333, 3.33333333, 3.33333333, 3.33333333,
        3.        , 5.        , 2.        ],
       [2.        , 3.        , 3.        , 3.        , 3.        ,
        3.        , 3.        , 4.        ],
       [4.5       , 4.5       , 4.5       , 4.5       , 4.5       ,
        5.        , 4.      

In [4]:
from numpy.linalg import svd
U, S, VT = svd(R_filled, full_matrices=False)
S

array([28.92699408,  4.8713192 ,  2.8518037 ,  2.2393418 ,  1.45939796,
        0.75840416,  0.29929804,  0.17161042])

In [17]:
k = 3
S_k = np.diag(S[:k])
U_k = U[:, :k]
VT_k = VT[:k]
R_hat = U_k @ S_k @ VT_k
R_hat_clipped = np.round(np.clip(R_hat, 1, 5), 2)
R_hat_clipped

array([[4.69, 3.25, 2.86, 1.  , 2.64, 2.75, 2.89, 2.18],
       [3.71, 2.66, 2.52, 1.  , 2.26, 2.37, 2.24, 2.04],
       [1.  , 1.67, 2.39, 4.26, 2.4 , 2.34, 2.58, 2.78],
       [4.64, 4.42, 4.83, 4.11, 4.46, 4.57, 4.46, 4.54],
       [3.32, 3.69, 4.36, 4.42, 3.99, 4.08, 3.81, 4.33],
       [2.32, 2.19, 3.07, 2.18, 2.31, 2.55, 1.02, 3.08],
       [3.33, 3.35, 2.83, 3.21, 3.32, 3.15, 5.  , 2.45],
       [2.35, 2.72, 3.3 , 3.46, 3.  , 3.07, 2.78, 3.33],
       [4.41, 4.34, 4.91, 4.31, 4.46, 4.59, 4.25, 4.7 ],
       [1.44, 1.84, 1.81, 2.47, 2.02, 1.93, 2.78, 1.76]])

In [18]:
for i, u in enumerate(R_hat_clipped):
    u_sorted = [n for n in np.argsort(u)[::-1] if R[i][n] == 0]
    print(f"Usuário {i+1}: Recomendamos os filmes F{u_sorted[0]+1} (nota prevista {u[u_sorted[0]]}) e F{u_sorted[1]+1} ({u[u_sorted[1]]})")

Usuário 1: Recomendamos os filmes F7 (nota prevista 2.89) e F3 (2.86)
Usuário 2: Recomendamos os filmes F2 (nota prevista 2.66) e F3 (2.52)
Usuário 3: Recomendamos os filmes F8 (nota prevista 2.78) e F7 (2.58)
Usuário 4: Recomendamos os filmes F1 (nota prevista 4.64) e F6 (4.57)
Usuário 5: Recomendamos os filmes F4 (nota prevista 4.42) e F3 (4.36)
Usuário 6: Recomendamos os filmes F8 (nota prevista 3.08) e F6 (2.55)
Usuário 7: Recomendamos os filmes F2 (nota prevista 3.35) e F1 (3.33)
Usuário 8: Recomendamos os filmes F4 (nota prevista 3.46) e F6 (3.07)
Usuário 9: Recomendamos os filmes F3 (nota prevista 4.91) e F8 (4.7)
Usuário 10: Recomendamos os filmes F4 (nota prevista 2.47) e F5 (2.02)
